In [ ]:
load_ext jupyter_black

In [ ]:
import os
import pandas as pd
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ]
}

In [ ]:
for dataset in demographics:
    df = pd.read_pickle(f"data/{dataset + '_utterances'}_linguistic.gz").drop(
        columns=[
            "s_neutral_model_response",
            "s_neutral_user_prompt",
            "model_response_liwc_Segment",
            "user_prompt_liwc_Segment",
        ],
        errors="ignore",
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df:
            df[c] = df[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df = df.rename(columns={"gpt_description": "topic"})
    group_cols = ["conversation_id"] + demographics[dataset] + ["topic"]
    df = (
        df.groupby(group_cols)[
            [
                c
                for c in df.columns
                if ("model_response" in c or "user_prompt" in c)
                and (c not in ["model_response", "user_prompt"])
            ]
        ]
        .mean()
        .reset_index()
    )

    for demographic in tqdm(demographics[dataset]):
        plot_df = (
            df.groupby([demographic, "topic"], as_index=False)
            .count()
            .rename(columns={"conversation_id": "count"})
        )
        plot_df = (
            pd.pivot_table(plot_df, "count", "topic", demographic).fillna(0).astype(int)
        )
        fig = plt.figure()
        sns.heatmap(plot_df, annot=True, fmt="d")
        locs, labels = plt.yticks()
        new_labels = []
        for label, row in zip(labels, plot_df.iterrows()):
            pvalue = chi2_contingency(
                np.vstack(
                    [
                        row[1].to_numpy(),
                        plot_df.sum(axis=0).to_numpy() - row[1].to_numpy(),
                    ]
                )
            ).pvalue
            if pvalue < 0.05:
                new_labels.append(label.get_text() + " *")
            else:
                new_labels.append(label)
        plt.yticks(locs, new_labels)

        plt.title(demographic)
        plt.savefig(
            f"figures_topic_demo_corr/chisquared_{dataset}_{demographic.replace(' ','')}.pdf",
            bbox_inches="tight",
        )
        plt.show()